In [0]:
-- Question 2 manual trace: Bronze -> Silver -> Gold
-- Test cases:
--   1. Normal quarterly: PRS30006011
--   2. Incomplete year:  PRS30006032
--   3. Q05 only:         PRS30006081
--   4. Tied best years:  PRS30006221


-- ============================================================
-- 1. BRONZE: roll source rows up to series/year
-- ============================================================
-- row explanation in a sentence:
-- for this series id, for Manufacturing, for Seasonal Adjusted Employment, % Change from same quarter 1 year ago,
-- for all workers, the best year was 2022 with a yearly value of 16.4
SELECT * FROM rearc.gold.bls_series_best_year;

WITH test_cases AS (
    SELECT * FROM VALUES
        ('normal_quarterly', 'PRS30006011', 2022),
        ('incomplete_year',  'PRS30006032', 1994),
        ('incomplete_year',  'PRS30006032', 2026),
        ('q05_only',         'PRS30006081', 2021),
        ('q05_only',         'PRS30006081', 2022),
        ('tied_best_years',  'PRS30006221', 2021),
        ('tied_best_years',  'PRS30006221', 2022)
    AS t(test_case, series_id, year)
)

SELECT
    t.test_case,
    b.series_id,
    CAST(b.year AS INT) AS year,
    -- count the number of (valid) quarters
    COUNT(CASE
        WHEN b.period IN ('Q01', 'Q02', 'Q03', 'Q04') THEN 1
    END) AS quarter_count,
    -- sum the values for each "normal" (q1-q4) quarter
    SUM(CASE
        WHEN b.period IN ('Q01', 'Q02', 'Q03', 'Q04')
        THEN CAST(b.value AS DOUBLE)
    END) AS quarterly_sum,
    -- MAX() lets us surface that single value alongside the other aggregates
    -- without adding period to GROUP BY clause
    MAX(CASE
        WHEN b.period = 'Q05'
        THEN CAST(b.value AS DOUBLE)
    END) AS q05_value
FROM rearc.bronze.bls_pr_data_1_alldata b
JOIN test_cases t
    ON b.series_id = t.series_id
   AND CAST(b.year AS INT) = t.year
GROUP BY
    t.test_case,
    b.series_id,
    CAST(b.year AS INT)
ORDER BY
    t.test_case,
    year;


-- ============================================================
-- 2. SILVER: same rollup; numbers should match Bronze
-- ============================================================

WITH test_cases AS (
    SELECT * FROM VALUES
        ('normal_quarterly', 'PRS30006011', 2022),
        ('incomplete_year',  'PRS30006032', 1994),
        ('incomplete_year',  'PRS30006032', 2026),
        ('q05_only',         'PRS30006081', 2021),
        ('q05_only',         'PRS30006081', 2022),
        ('tied_best_years',  'PRS30006221', 2021),
        ('tied_best_years',  'PRS30006221', 2022)
    AS t(test_case, series_id, year)
)

SELECT
    t.test_case,
    s.series_id,
    s.year,
    COUNT(CASE
        WHEN s.period IN ('Q01', 'Q02', 'Q03', 'Q04') THEN 1
    END) AS quarter_count,
    SUM(CASE
        WHEN s.period IN ('Q01', 'Q02', 'Q03', 'Q04')
        THEN s.value
    END) AS quarterly_sum,
    MAX(CASE
        WHEN s.period = 'Q05'
        THEN s.value
    END) AS q05_value
FROM rearc.silver.fct_bls_pr_data_1_alldata s
JOIN test_cases t
    ON s.series_id = t.series_id
   AND s.year = t.year
GROUP BY
    t.test_case,
    s.series_id,
    s.year
ORDER BY
    t.test_case,
    s.year;


-- ============================================================
-- 3. GOLD: inspect final winning rows
-- ============================================================

WITH test_cases AS (
    SELECT * FROM VALUES
        ('normal_quarterly', 'PRS30006011'),
        ('incomplete_year',  'PRS30006032'),
        ('q05_only',         'PRS30006081'),
        ('tied_best_years',  'PRS30006221')
    AS t(test_case, series_id)
)

SELECT
    t.test_case,
    g.series_id,
    g.value_method,
    g.best_year,
    g.yearly_value,
    g.human_readable_label
FROM rearc.gold.bls_series_best_year g
JOIN test_cases t
    ON g.series_id = t.series_id
ORDER BY
    t.test_case,
    g.best_year;
